In [ ]:
pip install sentence_transformers huggingface langchain langchain_community pinecone pinecone-client pinecone-notebooks pypdf langchain-huggingface chromadb rank_bm25 langchain-groq pinecone-text langchain_pinecone

In [ ]:
pip install sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

In [3]:
from langchain_community.document_loaders import PyPDFLoader
docpath = "/content/drive/MyDrive/VectorSave/cardiology/cardiology-explained.pdf"
loader = PyPDFLoader(docpath)

In [4]:
docs = loader.load()

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.chroma import Chroma

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500 ,chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
len(chunks)

1060

In [7]:
from langchain.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings)

In [8]:
from langchain_community.retrievers import BM25Retriever
keyword_retriever = BM25Retriever.from_documents(chunks, k=3)     # BM25 will be the keyword retriver or sparse retriever
vectorstore_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})    # Dense Vector embeddings will be retrived using vectorstore retriever
                                                                            # k is set to 3 as it will retrieve top 3 documents

In [9]:
from langchain.retrievers.ensemble import EnsembleRetriever         # Ensemble Retriver

ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retriever,
                                                   keyword_retriever],
                                       weights=[0.7, 0.3])                # Alpha values or weights be assigned as 0.7 and 0.3 for this use-case

In [ ]:
import nltk         #installing more dependencies
nltk.download('punkt_tab')

In [11]:
from sentence_transformers import CrossEncoder

def get_relevance_scores(question, documents):
    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    pairs = [[question, doc.page_content] for doc in documents]
    scores = ce.predict(pairs)
    return scores

In [12]:
def sort_and_display_documents(scores, documents, display_limit=200):

    # Pair scores with documents
    scored_docs = [(score, doc) for score, doc in zip(scores, documents)]

    # Sort by scores in descending order
    sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)

    # Extract the reordered documents
    reordered_docs = [doc for _, doc in sorted_docs]

    # Print scores, source, page numbers, document content
    print("Documents Sorted by Scores:\n")
    for score, doc in sorted_docs:
        # Extract file name from source path
        source_file_name = os.path.basename(doc.metadata.get('source', 'N/A'))

        print(f"Score: {score}, \nSource: {source_file_name}, Page: {doc.metadata.get('page', 'N/A')}, \nDoc:\n{doc.page_content[:display_limit]}\n\n")

    return reordered_docs

In [13]:
from langchain_core.output_parsers import StrOutputParser

In [14]:
def get_highest_ranked_document(scores, documents):

    if scores.size == 0 or not documents or len(scores) != len(documents):
        raise ValueError("Scores and documents must be non-empty and of the same length.")

    # Pair scores with documents and find the maximum by score
    highest_ranked = max(zip(scores, documents), key=lambda x: x[0])

    # Return the document with the highest score
    return highest_ranked[1]

In [15]:
def get_best_doc(question, documents):
    scores = get_relevance_scores(question, documents)
    return get_highest_ranked_document(scores, documents)

In [16]:
def full_retrieval(question):
    docs = ensemble_retriever.invoke(question)
    best_doc = get_best_doc(question, docs)
    return best_doc

In [17]:
from langchain_groq import ChatGroq

groqllm=ChatGroq(groq_api_key=userdata.get('GROQ'),
                 model_name= "groq/compound")

In [36]:
from langchain.prompts import ChatPromptTemplate

multi_template = """You are a helpful assistant that generates exactly 4 search queries expanding on a given topic.\n
    "Input question: {question}\n\n"
    "Output only a valid JSON array of 4 strings representing the search queries.\n"
    "Do not include any explanations, markdown, or additional text. Do not try to answer the question. only the raw JSON array."""

multi_prompt = ChatPromptTemplate.from_template(multi_template)

In [37]:
generate_queries = ( multi_prompt | groqllm | StrOutputParser() | (lambda x: x.split("\n")))

In [38]:
multi_query1 = "what are the classic symptoms of angina pectoris?"

In [39]:
multi_questions1 = generate_queries.invoke({"question":multi_query1})

In [40]:
multi_questions1

['["what are the common signs of angina pectoris","classic symptoms of stable angina pectoris","symptoms of unstable angina pectoris","how to identify angina pectoris symptoms"]']

In [43]:
multi_questions2 = generate_queries.invoke({"question":"what is a heart murmur?"})

In [44]:
multi_questions2

['["what are the causes of a heart murmur","heart murmur symptoms and treatment","types of heart murmurs in adults","can a heart murmur be normal"]']

In [45]:
import json
import re

def parse_llm_queries(output):
    """
    Extracts up to 4 search queries from an LLM output.
    Handles cases where:
      - output is a list of strings or a single string
      - JSON array is wrapped in quotes (e.g. '["..."]')
      - JSON code blocks are used
    Returns: list of 4 query strings
    """

    # Step 1: Normalize input
    if isinstance(output, list):
        text = "\n".join(output)
    elif isinstance(output, str):
        text = output
    else:
        raise TypeError("Input must be a string or list of strings.")

    text = text.strip()

    # Step 2: Extract JSON array (with or without code fences)
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if not match:
        raise ValueError("No JSON array found in the output.")

    json_content = match.group(0).strip()

    # Step 3: Parse JSON safely (handle double-encoded cases)
    try:
        data = json.loads(json_content)
        if isinstance(data, str):  # JSON string containing JSON array
            data = json.loads(data)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON content: {e}")

    # Step 4: Validate list structure
    if not isinstance(data, list) or not all(isinstance(q, str) for q in data):
        raise ValueError("Parsed content is not a list of strings.")

    # Step 5: Ensure exactly 4 queries
    return data[:4]


In [46]:
sub_queries = parse_llm_queries(multi_questions2)
print(sub_queries)

['what are the causes of a heart murmur', 'heart murmur symptoms and treatment', 'types of heart murmurs in adults', 'can a heart murmur be normal']


In [47]:
from langchain_core.prompts import PromptTemplate
gqtemplate = """Answer the question :
{question}
Based on the following context :
{context}

"""
gqprompt = PromptTemplate(
    template=gqtemplate,
    input_variables=["context", "question"],  # The context will be the top retrieved document and the question is the user query

)

In [48]:
groqchain = gqprompt | groqllm | StrOutputParser()

In [49]:
def groq_test(question):
  result = groqchain.invoke({"context": full_retrieval(question), "question": question})
  return result

In [50]:
def answer_all(query):
  multi_questions = generate_queries.invoke({"question":query})
  cleaned_questions = parse_llm_queries(multi_questions)
  rag_answers= []
  for q in cleaned_questions:
    answer = groqchain.invoke({"context": full_retrieval(q), "question": q})
    rag_answers.append(answer)
  return query,rag_answers, cleaned_questions

In [ ]:
query1,rag_answers1, cleaned_questions1 = answer_all(multi_query1)

In [52]:
query1

'what are the classic symptoms of angina pectoris?'

In [53]:
rag_answers1

['**Answer – Common signs (symptoms) of angina pectoris**\n\nThe source you gave (a short excerpt from a cardiology textbook) mentions the term “angina pectoris” but does not list its clinical manifestations.\u202fTo answer the question we rely on standard medical descriptions of angina, which are consistent across reputable sources (Mayo Clinic, Healthdirect, Penn Medicine, etc.).\n\n| Symptom | Typical description |\n|---------|----------------------|\n| **Chest discomfort/pain** | A pressure, heaviness, tightness, squeezing, burning or “crushing” sensation under the breastbone. Often triggered by exertion or emotional stress and relieved by rest or nitroglycerin. |\n| **Radiating pain** | May spread to the **arms (usually left), shoulders, back, neck, jaw, or even the ears**. |\n| **Shortness of breath (dyspnea)** | Often accompanies the chest pain, especially during activity. |\n| **Fatigue / weakness** | A feeling of being unusually tired or weak, sometimes described as “extreme t

In [54]:
cleaned_questions1

['what are the common signs of angina pectoris',
 'classic symptoms of stable angina pectoris',
 'symptoms of unstable angina pectoris',
 'how to identify angina pectoris symptoms']

In [57]:
query2,rag_answers2, cleaned_questions2 = answer_all("what is the most common cause of ischemic heart disease?")

In [58]:
cleaned_questions2

['what are the risk factors for ischemic heart disease',
 'most common cause of ischemic heart disease',
 'how does atherosclerosis cause ischemic heart disease',
 'is high blood pressure a cause of ischemic heart disease']